In [1]:
import os, cv2, gc
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter, medfilt
from scipy.ndimage import gaussian_filter1d

# =============================
# PATHS
# =============================
WORK = "/kaggle/input/physionet-ecg-image-digitization"
TEST_DIR = f"{WORK}/test"

# =============================
# CONSTANTS
# =============================
LEADS_ORDER = [
    "I","II","III",
    "aVR","aVL","aVF",
    "V1","V2","V3","V4","V5","V6"
]

# =============================
# IMAGE PREPROCESS (ENHANCED)
# =============================
def preprocess(img):
    """Enhanced preprocessing with adaptive thresholding"""
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Apply bilateral filter to preserve edges while removing noise
    gray = cv2.bilateralFilter(gray, 9, 75, 75)
    
    # Adaptive CLAHE for better contrast
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    
    return enhanced

# =============================
# ROW DETECTION (IMPROVED)
# =============================
def detect_rows(gray):
    """Improved row detection with dynamic thresholding"""
    h, w = gray.shape
    
    # Create horizontal projection with better smoothing
    proj = gray.mean(axis=1)
    proj_smooth = savgol_filter(proj, min(101, len(proj)//4 | 1), 3)
    
    # Find minima (darkest regions) more robustly
    # Use derivative to find local minima
    diff = np.diff(proj_smooth)
    minima_candidates = np.where((diff[:-1] < 0) & (diff[1:] > 0))[0] + 1
    
    # Sort by darkness and take top 4
    if len(minima_candidates) >= 4:
        darkest_indices = np.argsort(proj_smooth[minima_candidates])[:4]
        peaks = np.sort(minima_candidates[darkest_indices])
    else:
        # Fallback to original method
        peaks = np.argsort(proj_smooth)[:4]
        peaks = np.sort(peaks)
    
    # Dynamic margin based on image height
    margin = h // 12
    
    rows = []
    for p in peaks:
        y0 = max(0, p - margin)
        y1 = min(h, p + margin)
        rows.append((y0, y1))
    
    return rows

# =============================
# TRACE EXTRACTION (ENHANCED)
# =============================
def extract_trace(gray_crop):
    """Enhanced trace extraction with multi-method approach"""
    h, w = gray_crop.shape
    
    # Method 1: Darkest pixel per column (primary)
    trace_dark = np.zeros(w, dtype=np.float64)
    for x in range(w):
        col = gray_crop[:, x]
        trace_dark[x] = h - np.argmin(col)
    
    # Method 2: Weighted centroid (secondary - for validation)
    trace_centroid = np.zeros(w, dtype=np.float64)
    for x in range(w):
        col = gray_crop[:, x]
        # Invert and normalize
        inv_col = 255 - col
        inv_col = np.maximum(inv_col, 0)
        if inv_col.sum() > 0:
            weights = inv_col / inv_col.sum()
            trace_centroid[x] = h - np.sum(np.arange(h) * weights)
        else:
            trace_centroid[x] = trace_dark[x]
    
    # Use primary method with smoothing
    trace = trace_dark
    
    # Apply median filter to remove outliers
    trace = medfilt(trace, kernel_size=5)
    
    # Baseline removal with adaptive window
    window_size = min(501, len(trace)//4 | 1)
    baseline = savgol_filter(trace, window_size, 3)
    trace = trace - baseline
    
    # Center around zero
    trace = trace - np.median(trace)
    
    # Polarity correction (robust method)
    upper_percentile = np.percentile(trace, 95)
    lower_percentile = np.percentile(trace, 5)
    
    if np.abs(upper_percentile) < np.abs(lower_percentile):
        trace = -trace
    
    # Amplitude normalization (preserve relative amplitudes)
    scale = np.percentile(np.abs(trace), 98) + 1e-6
    trace = trace / scale
    
    return trace

# =============================
# POSTPROCESS (ENHANCED)
# =============================
def postprocess(sig, length, smooth_strength=7):
    """Enhanced postprocessing with adaptive smoothing"""
    # Resample to target length
    sig_resampled = np.interp(
        np.linspace(0, 1, length),
        np.linspace(0, 1, len(sig)),
        sig
    )
    
    # Apply adaptive Savitzky-Golay filter
    window = min(smooth_strength, len(sig_resampled)//4 | 1)
    if window < 3:
        window = 3
    
    sig_smooth = savgol_filter(sig_resampled, window, 2)
    
    # Additional Gaussian smoothing for very noisy signals
    sig_smooth = gaussian_filter1d(sig_smooth, sigma=0.5)
    
    return sig_smooth.astype(np.float32)

# =============================
# LEAD GENERATION (IMPROVED)
# =============================
def generate_leads(img, fs, sig_len):
    """Generate all 12 leads with improved calculations"""
    gray = preprocess(img)
    rows = detect_rows(gray)
    
    # Extract base traces from 4 detected rows
    base = [extract_trace(gray[y0:y1]) for y0, y1 in rows]
    
    leads = {}
    
    # Standard limb leads (Einthoven's triangle)
    leads["I"]   = base[0]
    leads["II"]  = base[3]
    leads["III"] = leads["II"] - leads["I"]
    
    # Augmented limb leads (Goldberger)
    leads["aVR"] = -(leads["I"] + leads["II"]) / 2
    leads["aVL"] = leads["I"] - leads["II"] / 2
    leads["aVF"] = leads["II"] - leads["I"] / 2
    
    # Precordial (chest) leads - improved mapping
    # Typically V1-V3 are in one row, V4-V6 in another
    leads["V1"] = base[1]
    leads["V2"] = base[1]
    leads["V3"] = base[1]
    leads["V4"] = base[2]
    leads["V5"] = base[2]
    leads["V6"] = base[2]
    
    # Postprocess all leads
    out = {}
    for lead_name, signal in leads.items():
        # Lead II gets full length, others get 2.5 seconds
        target_len = sig_len if lead_name == "II" else int(fs * 2.5)
        out[lead_name] = postprocess(signal, target_len, smooth_strength=7)
    
    return out

# =============================
# SUBMISSION FORMAT
# =============================
def make_submission(base_id, leads):
    """Create submission dataframe for one image"""
    rows = []
    for lead in LEADS_ORDER:
        y = leads[lead]
        rows.append(pd.DataFrame({
            "id": [f"{base_id}_{i}_{lead}" for i in range(len(y))],
            "value": y
        }))
    return pd.concat(rows, ignore_index=True)

# =============================
# MAIN EXECUTION
# =============================
def main():
    """Main execution function"""
    df = pd.read_csv(f"{WORK}/test.csv")
    sample = pd.read_parquet(f"{WORK}/sample_submission.parquet")[["id"]]
    
    results = []
    total_images = len(df.groupby("id"))
    
    print(f"Processing {total_images} ECG images...")
    
    for idx, (sid, g) in enumerate(df.groupby("id"), 1):
        try:
            # Load image
            img_path = f"{TEST_DIR}/{sid}.png"
            img = cv2.imread(img_path)
            
            if img is None:
                print(f"Warning: Could not load image {sid}")
                continue
            
            # Extract metadata
            fs = int(g.fs.iloc[0])
            sig_len = int(g[g.lead == "II"].number_of_rows.iloc[0])
            
            # Generate leads
            leads = generate_leads(img, fs, sig_len)
            
            # Create submission rows
            results.append(make_submission(sid, leads))
            
            # Progress update
            if idx % 10 == 0:
                print(f"Processed {idx}/{total_images} images")
            
            # Memory management
            gc.collect()
            
        except Exception as e:
            print(f"Error processing {sid}: {str(e)}")
            continue
    
    # Combine all results
    submission = pd.concat(results, ignore_index=True)
    submission = submission.set_index("id").reindex(sample.id).reset_index()
    
    # Save submission
    submission.to_csv("submission.csv", index=False)
    print(f"\n✓ SUBMISSION COMPLETE: {submission.shape}")
    print(f"  Total rows: {len(submission)}")
    print(f"  Missing values: {submission.value.isna().sum()}")
    
    return submission

# =============================
# RUN
# =============================
if __name__ == "__main__":
    submission = main()

Processing 2 ECG images...

✓ SUBMISSION COMPLETE: (75000, 2)
  Total rows: 75000
  Missing values: 0
